# 02.1 — DoReFaNet ImageNet pe Raspberry Pi

**Țintă:** Raspberry Pi 4, 4 GB, Debian 13 ARM64 (`aarch64`). Urmați
[ghidul instalării](../raspberry_pi/README.md), apoi rulați testul de referință al modelului.

Se folosesc greutățile originale și biblioteca software ARM64 din proiect, fără FPGA.
Aceasta este o referință C++/HLS, nu un motor AI optimizat pentru Raspberry Pi; un apel poate dura minute.
Mesajul periodic indică doar timpul scurs. Întreruperea unei funcții native poate fi întârziată.
**Inferența reală pe Raspberry Pi nu a fost încă validată.**

Notebookul nu salvează automat imagini sau jurnale de rezultate. Jupyter poate salva ieșirile celulelor
în fișierul notebook; ștergeți ieșirile înainte de a salva dacă nu doriți să persiste.


## 1. Mediu

In [ ]:
import sys, gc, time, hashlib, pickle
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from threadpoolctl import threadpool_limits, threadpool_info

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "raspberry_pi/qnn_cpu.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository with raspberry_pi/start_jupyter.sh")
sys.path.insert(0, str(ROOT / "raspberry_pi"))
from qnn_cpu import DorefanetCPU, TinierYoloCPU, DarknetCPU, utils, snapshot, temporary_image, machine_info
if "blas_limit" in globals():
    blas_limit.restore_original_limits()
blas_limit = threadpool_limits(limits=1)
cv2.setNumThreads(1)
QNN_PARAM_DIR = ROOT / "qnn/params"
NOTEBOOK_DIR = ROOT / "notebooks"
print(machine_info())
print(threadpool_info())
BACKGROUND_CAMERA_STREAM = "uStreamer active for Home Assistant"  # record actual condition

SAMPLE_DIR = NOTEBOOK_DIR / "imagenet-samples"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"


## 2. Greutăți și model CPU

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the CPU reference...", flush=True)
classifier = DorefanetCPU()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 3. Aceeași imagine ca în PYNQ

In [ ]:
IMAGE_NAME = "n01484850_0.jpg"
image_path = SAMPLE_DIR / IMAGE_NAME
input_name = image_path.name
input_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
timings = {}
t0 = time.perf_counter()
img, img_class = classifier.load_image(str(image_path))
timings['preprocess_s'] = time.perf_counter() - t0
expected_index = synset_to_index.get(img_class)
print(input_name, input_sha256)
with Image.open(image_path) as opened:
    preview = opened.convert('RGB')
preview.thumbnail((480, 360))
display(preview)


## 4. Prima convoluție

In [ ]:
start = time.perf_counter()
conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
timings['first_cpu_s'] = time.perf_counter() - start
print("CPU first stage: {:.3f} s".format(
    timings['first_cpu_s']))

## 5. Straturile cuantizate pe CPU

In [ ]:
out_dim = net["merge4"]["output_dim"]
out_ch = net["merge4"]["output_channels"]
packing_start = time.perf_counter()
conv_output = classifier.get_accel_buffer(out_ch, out_dim)
conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
timings['packing_s'] = time.perf_counter() - packing_start
start = time.perf_counter()
classifier.inference(conv_input, conv_output)
timings['quantized_core_s'] = time.perf_counter() - start
unpack_start = time.perf_counter()
conv_output = classifier.postprocess_buffer(conv_output)
timings['unpacking_s'] = time.perf_counter() - unpack_start
quantized_sha256 = hashlib.sha256(np.ascontiguousarray(conv_output, dtype='<f4').tobytes()).hexdigest()
if not np.all(np.isfinite(conv_output)):
    raise ValueError("Non-finite accelerator output. Check model initialization.")
print("CPU reference call: {:.3f} s".format(
    timings['quantized_core_s']))

## 6. Straturi complet conectate

In [ ]:
scale = np.max(conv_output)
if scale <= 0:
    raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
fc_input = conv_output / scale
if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
    raise ValueError("Accelerator output size does not match the fully connected weights.")
start = time.perf_counter()
fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
fc1_out = utils.qrelu(fc1_out)
fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
timings['last_cpu_s'] = time.perf_counter() - start
print("CPU fully connected stages: {:.3f} s".format(
    timings['last_cpu_s']))

## 7. Top-5

In [ ]:
if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
    raise ValueError("Expected 1,000 finite classification scores.")
post_start = time.perf_counter()
out = utils.softmax(fct_out)
topn = utils.get_topn_indexes(out, 5)
timings['postprocess_s'] = time.perf_counter() - post_start
print("Correct class:", names[expected_index] if expected_index is not None else "unknown")
for index in topn:
    print("{:25s} {:7.2%}".format(names[int(index)], float(out[index])))
if expected_index is not None:
    print("Top-1:", "correct" if int(topn[0]) == expected_index else "wrong")
    print("Top-5:", "correct class included" if expected_index in topn
          else "correct class not included")
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(topn))
colors = ["green" if int(index) == expected_index else "steelblue" for index in topn]
ax.barh(positions, out[topn], color=colors)
ax.set_yticks(positions)
ax.set_yticklabels([names[int(index)] for index in topn])
ax.invert_yaxis()
ax.set_xlim(0, min(1.0, float(np.max(out[topn])) * 1.25))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Softmax score (axis scaled to the displayed predictions)")
ax.set_title("Top-5 predictions (green = known correct class)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)


### Compararea cu PYNQ
Folosiți aceeași imagine și aceleași greutăți. Comparați `quantized_core_s` cu timpul apelului
`classifier.inference(...)` din notebookul PYNQ. Acesta exclude împachetarea/dezîmpachetarea.
Suma de mai jos include preprocesarea și bufferele; graficele PYNQ 02/05 nu includ toate aceste etape.
Nu comparați aceste sume ca și cum ar avea aceleași limite.

Pentru verificare, comparați hash-ul intrării și hash-ul ieșirii cuantizate (în PYNQ folosiți
`conv_output` pentru 02 și `fpga_output` pentru 05):
```python
import hashlib
hashlib.sha256(np.ascontiguousarray(conv_output, dtype="<f4").tobytes()).hexdigest()
```
Rezultatele numerice ale etapelor float pot varia cu BLAS/CPU. Păstrați aceleași limite de threaduri,
faceți o încălzire și raportați mediana mai multor rulări. Notați temperatura, alimentarea și
serviciile active. uStreamer consumă resurse și trebuie menționat în rezultate.


In [ ]:
measurement = dict(input_name=input_name, input_sha256=input_sha256,
                   quantized_sha256=quantized_sha256, timings=dict(timings),
                   pipeline_sum_s=sum(timings.values()), machine=machine_info(),
                   background_camera_stream=BACKGROUND_CAMERA_STREAM)
for name, value in timings.items():
    print(f"{name}: {value:.4f} s")
print(f"Sum of measured stages: {measurement['pipeline_sum_s']:.4f} s")
print("Quantized output SHA256:", quantized_sha256)
PYNQ_QUANTIZED_CORE_S = None  # enter measured classifier.inference duration from PYNQ
if PYNQ_QUANTIZED_CORE_S is not None:
    if PYNQ_QUANTIZED_CORE_S <= 0:
        raise ValueError("PYNQ duration must be positive")
    print("CPU reference / FPGA call ratio:", timings['quantized_core_s'] / PYNQ_QUANTIZED_CORE_S)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(timings), list(timings.values()))
ax.invert_yaxis()
ax.set_xlabel("Measured seconds")
ax.set_title("Raspberry Pi CPU reference — one inference")
fig.tight_layout()
plt.show()
plt.close(fig)
# measurement stays in memory; no result JSON or capture file is saved.


## 8. Eliberare memorie

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")
blas_limit.restore_original_limits()
